TESTE REALIZADO NO MODELO OLLAMA PARA GERAÇÃO DOS EMBEDDINGS

In [65]:
import pdfplumber

caminho_pdf = r"C:\Users\Lucas\OneDrive\Desktop\ia\arquivos\Fundamentos-tecnicos-e-taticos-do-Futebol-FINAL.pdf"

def extrair_texto_pdf(caminho, pagina_inicial=9):
    textos = []

    with pdfplumber.open(caminho) as pdf:
        for pagina in pdf.pages[pagina_inicial:]:
            texto = pagina.extract_text()
            if texto:
                textos.append(texto)

    return "\n".join(textos)

texto_completo = extrair_texto_pdf(caminho_pdf, pagina_inicial=9)

In [66]:
import re

def limpar_texto(texto: str) -> str:
    texto = re.sub(r"\r\n", "\n", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)

    # remover espaços antes de quebra de linha
    texto = re.sub(r" *\n", "\n", texto)

    # remover múltiplos espaços novamente (segurança)
    texto = re.sub(r" {2,}", " ", texto)

    return texto.strip()

texto_limpo = limpar_texto(texto_completo)

In [67]:
def criar_chunks(texto: str, tamanho: int = 400, overlap: int = 80):
    chunks = []
    inicio = 0
    tamanho_texto = len(texto)

    while inicio < tamanho_texto:
        fim = inicio + tamanho

        # evita cortar palavra no meio
        if fim < tamanho_texto:
            while fim < tamanho_texto and texto[fim] != " ":
                fim += 1

        chunk = texto[inicio:fim].strip()

        if chunk:
            chunks.append(chunk)

        inicio += tamanho - overlap

    return chunks


chunks = criar_chunks(texto_limpo, tamanho=400, overlap=80)

print(f"Quantidade de chunks: {len(chunks)}")
print(chunks[0][:500])

Quantidade de chunks: 395
Fundamentos técnicos e táticos do futebol
INTRODUÇÃO
O futebol é um esporte de enfrentamentos, de
confrontos, no qual duas equipes, formadas por 11 jogadores,
sendo um deles o goleiro, utilizando principalmente os pés e a
demais partes do corpo (exceto braços e mãos) para movimentar
a bola em direção ao campo adversário, com o objetivo de
colocá-la dentro da meta adversária.
É um assunto fascinante


In [68]:
import ollama

def gerar_emb(textos, batch_size=32):
    todos_embeddings = []

    for i in range(0, len(textos), batch_size):
        lote = textos[i:i + batch_size]

        response = ollama.embed(
            model="embeddinggemma:latest",
            input=lote
        )

        if isinstance(response, dict):
            embeddings = response["embeddings"]
        else:
            embeddings = response.embeddings

        todos_embeddings.extend(embeddings)

    return todos_embeddings


emb_doc = gerar_emb(chunks, batch_size=32)

In [69]:
import pickle

import pickle

data = {
    "chunks": chunks,
    "embeddings": emb_doc
}

with open("base_rag.pkl", "wb") as f:
    pickle.dump(data, f)

In [70]:
# %% 🚀 CRIAÇÃO DA BASE OTIMIZADA

base = [
    {
        "id": i,
        "texto": chunk,
        "embedding": emb
    }
    for i, (chunk, emb) in enumerate(zip(chunks, emb_doc))
]

In [71]:
import pickle
import numpy as np

with open("base_rag.pkl", "rb") as f:
    data = pickle.load(f)

chunks = data["chunks"]
emb_doc = data["embeddings"]

matriz_embeddings = np.array(emb_doc, dtype=float)
textos = chunks

matriz_norm = matriz_embeddings / np.linalg.norm(
    matriz_embeddings, axis=1, keepdims=True
)

In [72]:
# %% 🚀 BUSCA FINAL (ÚNICA E CORRETA)

def buscar_chunks_relevantes(pergunta, top_k=3):
    emb_perg = np.array(gerar_emb([pergunta])[0])

    # normaliza embedding da pergunta
    emb_perg = emb_perg / np.linalg.norm(emb_perg)

    # similaridade vetorizada
    scores = np.dot(matriz_norm, emb_perg)

    # pegar top_k
    top_indices = np.argsort(scores)[-top_k:][::-1]

    resultados = [
        {
            "id": int(idx),
            "texto": textos[idx],
            "score": float(scores[idx])
        }
        for idx in top_indices
    ]

    return resultados

In [73]:
pergunta = "O que é drible?"

resultados = buscar_chunks_relevantes(pergunta)

for r in resultados:
    print(r["score"], r["texto"][:200])

0.5586610090430568 a intenção
de superar o oponente. Segundo Tenroller (2004), trata-se de
uma série de movimentos e ações que culmina com a superação
do adversário e a sequência da jogada com a posse da bola. Para
Cost
0.5495010646712772 enso.
 Não impulsionar a bola suavemente.
 Tronco muito rígido; não inclinando-se levemente para
frente.
 Movimentos do corpo durante a corrida não ritmados.
 Olhar fixado na bola e não no campo d
0.5024374752946364 apoio, posição do corpo, equilíbrio),
velocidade de execução, agilidade, local do campo onde será
realizado o drible e situação de jogo.
Finta
Para Leal (2000), é a ação realizada imediatamente
antes 


In [74]:
def montar_contexto(resultados):
    return "\n\n".join(
        f"{item['texto']}"
        for item in resultados
    )

In [75]:
# %% 🚀 RESPOSTA FINAL OTIMIZADA

def responder_pergunta(pergunta, top_k=3, modelo_llm="llama3.2:latest"):
    resultados = buscar_chunks_relevantes(pergunta, top_k=top_k)
    contexto = montar_contexto(resultados)

    prompt = f"""
    Responda à pergunta usando apenas o contexto fornecido.

    Regras:
    - Não mencione o contexto, texto ou fonte.
    - Não use frases como "de acordo com", "segundo o texto" ou similares.
    - Não invente informações.
    - Não use conhecimento externo.
    - Se não houver resposta no contexto, diga que não encontrou.
    - Responda de forma natural, objetiva e completa.

    Contexto:
    {contexto}

    Pergunta:
    {pergunta}

    Resposta:
    """

    resposta = ollama.chat(
        model=modelo_llm,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    if isinstance(resposta, dict):
        return resposta["message"]["content"], resultados
    else:
        return resposta.message.content, resultados

In [77]:
# %% 🚀 TESTE FINAL

pergunta = "O que é finta?"

resposta, fontes = responder_pergunta(
    pergunta,
    top_k=4,
    modelo_llm="llama3.2:latest"
)

print("RESPOSTA:\n")
print(resposta)

RESPOSTA:

A finta é uma ação realizada imediatamente antes do drible, do passe ou da finalização com o objetivo de iludir o adversário, sugerindo um movimento que não é o verdadeiro, a fim de obter uma vantagem ou criar um espaço inexistente.
